In [3]:
from text_classification.data import load_data, split_indices

df_train, df_test = load_data('../data/')

SEED=42 

In [2]:
from text_classification.preprocessing import preprocess_text

versions = {
    # удаление спец символов/пунктуации + lowercase 
    "v1_basic": {},
    # удаление спец символов + lowercase + stopwords 
    "v2_no_stopwords": {
        "remove_stopwords": True
    },
    # удаление спец символов/пунктуации + lowercase + лемматизация 
    "v3_lemma": {
        "normalization": "lemma"
    },
    # удаление спец символов/пунктуации + lowercase + stopwords + лемматизация
    "v4_lemma_no_stopwords": {
        "normalization": "lemma",
        "remove_stopwords": True
    },
    # удаление спец символов/пунктуации + lowercase + стемминг 
    "v5_stem": {
        "normalization": "stem"
    },
    # только удаление спец символов/пунктуации 
    "v6_original_case": {
        "lowercase": False
    }
}

df_versions_train = df_train.copy()
df_versions_test = df_test.copy()


for name, params in versions.items():
    df_versions_train[name] = df_train["text"].apply(
        lambda text: preprocess_text(text, **params)
    )
    
for name, params in versions.items():
    df_versions_test[name] = df_test["text"].apply(
        lambda text: preprocess_text(text, **params)
    )

In [ ]:
tfidf_dataset = {}
vectorizers = {}

y = df_versions_train["target"]

train_idx, val_idx = split_indices(train=df_train, seed=SEED)

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

for version in versions: 
    
    vectorizer = TfidfVectorizer(
        analyzer="word",
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 1),
        min_df=2,
        max_df=0.95,
        max_features=10000,
        norm="l2",
        sublinear_tf=True
    )
    
    X_train = vectorizer.fit_transform(df_versions_train[version].iloc[train_idx])
    X_val = vectorizer.transform(df_versions_train[version].iloc[val_idx])
    X_test = vectorizer.transform(df_versions_test[version])
    
    vectorizers[version] = vectorizer
    
    tfidf_dataset[version] = {
        'X_train': X_train, 
        'X_val': X_val, 
        'X_test': X_test
    }